In [0]:
-- Databricks SQL Editor: NYC Yellow Taxi (2016-02) – Bronze/Silver/Gold
-- Author: Yuri
-- Notes:
--  - Catalog/schema: main.nyc_taxi
--  - Tables:
--      BRONZE: main.nyc_taxi.yellow_tripdata_2016_02_bronze
--      SILVER: main.nyc_taxi.yellow_tripdata_2016_02_silver
--      GOLD  : main.nyc_taxi.yellow_trip_2016_02_gold_daily

/****************************
 * 0) Contexto do Catálogo  *
 ****************************/
USE CATALOG main;
USE nyc_taxi;

-- (Opcional) Ver todos os objetos do schema
-- SHOW TABLES;

/****************************
 * 1) BRONZE – inspeção     *
 ****************************/
-- 1.1 Primeiras 10 linhas
SELECT *
FROM yellow_tripdata_2016_02_bronze
LIMIT 10;

-- 1.2 Contagem de linhas
SELECT COUNT(*) AS total_rows
FROM yellow_tripdata_2016_02_bronze;

-- 1.3 Esquema da tabela
DESCRIBE TABLE EXTENDED yellow_tripdata_2016_02_bronze;

-- 1.4 Amostra estratificada (se houver coluna vendorid)
SELECT *
FROM yellow_tripdata_2016_02_bronze
TABLESAMPLE (5 PERCENT)
LIMIT 100;

/****************************
 * 2) SILVER – qualidade    *
 ****************************/
-- 2.1 Estatísticas básicas
SELECT
  COUNT(*)                              AS n_trips,
  AVG(trip_distance)                    AS avg_distance,
  AVG(fare_amount)                      AS avg_fare,
  AVG(tip_rate)                         AS avg_tip_rate,
  AVG(avg_speed_mph)                    AS avg_speed_mph
FROM yellow_tripdata_2016_02_silver;

-- 2.2 Distribuição diária
SELECT
  pickup_date,
  COUNT(*)                  AS trips,
  AVG(trip_distance)        AS avg_miles,
  AVG(fare_amount)          AS avg_fare,
  AVG(tip_rate)             AS avg_tip_rate
FROM yellow_tripdata_2016_02_silver
GROUP BY pickup_date
ORDER BY pickup_date;

-- 2.3 Top 10 corridas por taxa de gorjeta
SELECT
  pickup_ts, dropoff_ts, passenger_count,
  trip_distance, fare_amount, tip_amount, tip_rate
FROM yellow_tripdata_2016_02_silver
WHERE tip_rate IS NOT NULL
ORDER BY tip_rate DESC
LIMIT 10;

-- 2.4 Histograma por hora do dia (média de distância e contagem)
SELECT
  hour,
  COUNT(*)           AS trips,
  ROUND(AVG(trip_distance), 2) AS avg_miles
FROM yellow_tripdata_2016_02_silver
GROUP BY hour
ORDER BY hour;

-- 2.5 Potenciais outliers de velocidade (acima de 60 mph em média)
SELECT
  pickup_ts, dropoff_ts, trip_distance, trip_minutes, avg_speed_mph
FROM yellow_tripdata_2016_02_silver
WHERE avg_speed_mph IS NOT NULL AND avg_speed_mph > 60
ORDER BY avg_speed_mph DESC
LIMIT 50;

-- 2.6 Distribuição de forma de pagamento
SELECT
  payment_type,
  COUNT(*) AS trips,
  ROUND(AVG(fare_amount), 2) AS avg_fare,
  ROUND(AVG(tip_rate), 4) AS avg_tip_rate
FROM yellow_tripdata_2016_02_silver
GROUP BY payment_type
ORDER BY trips DESC;

-- 2.7 Vendor x Forma de pagamento (pivot simples)
SELECT *
FROM (
  SELECT vendor_id, payment_type, 1 AS cnt
  FROM yellow_tripdata_2016_02_silver
) src
PIVOT (
  SUM(cnt) FOR payment_type IN (1,2,3,4,5,6)
)
ORDER BY vendor_id;

-- 2.8 Janela: ranking por gorjeta por dia
WITH ranked AS (
  SELECT
    pickup_date, pickup_ts, dropoff_ts, passenger_count,
    fare_amount, tip_amount, tip_rate,
    DENSE_RANK() OVER (PARTITION BY pickup_date ORDER BY tip_rate DESC) AS rnk
  FROM yellow_tripdata_2016_02_silver
  WHERE tip_rate IS NOT NULL
)
SELECT *
FROM ranked
WHERE rnk <= 5
ORDER BY pickup_date, rnk;

-- 2.9 Partição (verificação): linhas de 2016-02
SELECT COUNT(*) AS silver_rows_feb
FROM yellow_tripdata_2016_02_silver
WHERE year = 2016 AND month = 2;

/****************************
 * 3) GOLD – agregações     *
 ****************************/
-- 3.1 Visualizar 10 primeiras
SELECT *
FROM yellow_trip_2016_02_gold_daily
LIMIT 10;

-- 3.2 Métricas diárias resumidas
SELECT
  year, month, pickup_date,
  SUM(n_trips)        AS total_trips,
  ROUND(AVG(avg_fare), 2)     AS avg_fare,
  ROUND(AVG(avg_tip_rate), 4) AS avg_tip_rate
FROM yellow_trip_2016_02_gold_daily
GROUP BY year, month, pickup_date
ORDER BY pickup_date;

-- 3.3 Top 10 dias com maior taxa média de gorjeta
SELECT
  pickup_date,
  vendor_id,
  payment_type,
  n_trips,
  ROUND(avg_fare, 2)     AS avg_fare,
  ROUND(avg_tip_rate, 4) AS avg_tip_rate
FROM yellow_trip_2016_02_gold_daily
ORDER BY avg_tip_rate DESC
LIMIT 10;

-- 3.4 Vendor x Payment – ranking por dia
WITH by_day AS (
  SELECT
    pickup_date, vendor_id, payment_type,
    n_trips,
    avg_fare,
    avg_tip_rate,
    ROW_NUMBER() OVER (PARTITION BY pickup_date ORDER BY avg_tip_rate DESC) AS rnk
  FROM yellow_trip_2016_02_gold_daily
)
SELECT *
FROM by_day
WHERE rnk <= 3
ORDER BY pickup_date, rnk;

-- 3.5 Foco em fevereiro/2016 apenas (partições)
SELECT *
FROM yellow_trip_2016_02_gold_daily
WHERE year = 2016 AND month = 2
ORDER BY pickup_date, vendor_id, payment_type;

/**********************************
 * 4) Manutenção (opcional) Delta *
 **********************************/
--Compactação de arquivos pequenos (requer permissões UC adequadas)
OPTIMIZE yellow_tripdata_2016_02_silver;
OPTIMIZE yellow_trip_2016_02_gold_daily;

--Remoção de arquivos antigos (retenção de 7 dias = 168 horas)
VACUUM yellow_tripdata_2016_02_silver RETAIN 168 HOURS;
VACUUM yellow_trip_2016_02_gold_daily RETAIN 168 HOURS;

--EXPLAIN/plan (debug):
EXPLAIN SELECT * FROM yellow_tripdata_2016_02_silver WHERE year = 2016 AND month = 2;
